# zip
`zip` is how Python pairs multiple iterables into aligned tuples. In real systems, that matters for joining labels to values, columns to headers, services to regions, and input streams that are supposed to stay synchronized.

## 1. Problem First
Why does this matter in real systems?
- CSV headers need to align with row values.
- Monitoring labels often pair names with metric values.
- Silent misalignment between two lists can corrupt downstream output.

In [1]:
services = ["api", "worker"]
regions = ["us", "eu"]
for service, region in zip(services, regions):
    print(service, region)

api us
worker eu


## 2. Minimal Concept
Core syntax:
- `zip(iterable1, iterable2)`
- `zip(iterable1, iterable2, iterable3)`
- It yields tuples made from matching positions.

## 3. Mental Model
How Python actually behaves
`zip` walks all input iterables in parallel and stops at the shortest one. That default is convenient, but it can also hide missing data if you expected all sequences to be the same length.

In [2]:
headers = ["host", "port"]
values = ["api.internal", 8080]
print(list(zip(headers, values)))

[('host', 'api.internal'), ('port', 8080)]


## 4. Internal Mechanics
`zip` returns an iterator in Python 3. It does not build the full paired result unless you convert it to a list. Its stop-at-shortest behavior is one of the most important mechanical details to remember.

In [3]:
import dis

def pair_values(keys, values):
    for key, value in zip(keys, values):
        print(key, value)

dis.dis(pair_values)
print(list(zip([1, 2, 3], ["a", "b"])))

  3           0 RESUME                   0

  4           2 LOAD_GLOBAL              1 (NULL + zip)
             12 LOAD_FAST                0 (keys)
             14 LOAD_FAST                1 (values)
             16 CALL                     2
             24 GET_ITER
        >>   26 FOR_ITER                17 (to 64)
             30 UNPACK_SEQUENCE          2
             34 STORE_FAST               2 (key)
             36 STORE_FAST               3 (value)

  5          38 LOAD_GLOBAL              3 (NULL + print)
             48 LOAD_FAST                2 (key)
             50 LOAD_FAST                3 (value)
             52 CALL                     2
             60 POP_TOP
             62 JUMP_BACKWARD           19 (to 26)

  4     >>   64 END_FOR
             66 RETURN_CONST             0 (None)
[(1, 'a'), (2, 'b')]


## 5. Edge Cases
Where it breaks:
- If one iterable is shorter, extra data in the longer one is silently ignored.
- Paired values can look correct while data loss has already happened.
- Reusing a consumed zip iterator returns nothing.

In [4]:
keys = ["host", "port", "debug"]
values = ["api.internal", 8080]
pairs = list(zip(keys, values))
print(pairs)

z = zip([1, 2], [3, 4])
print(list(z))
print(list(z))

[('host', 'api.internal'), ('port', 8080)]
[(1, 3), (2, 4)]
[]


## 6. Performance Thinking
- `zip` is O(1) extra memory as an iterator.
- Total iteration cost is O(n) over the shortest iterable.
- It is often cleaner and safer than manual index-based parallel iteration.
- If alignment is critical, validation cost may matter more than pairing cost.

## 7. Real Use Case
A parser can combine CSV headers with row values to form structured records.

In [5]:
headers = ["service", "status", "latency_ms"]
row = ["api", "ok", 120]
record = dict(zip(headers, row))
print(record)

{'service': 'api', 'status': 'ok', 'latency_ms': 120}


## 8. Anti-Pattern
What beginners do wrong:
- Assume `zip` will complain about unequal lengths.
- Use manual index loops when direct pairing would be clearer.
- Forget that a zip object is a one-time iterator.

In [6]:
left = [10, 20, 30]
right = [1, 2]
for i in range(len(right)):
    print(left[i], right[i])

for pair in zip(left, right):
    print(pair)

10 1
20 2
(10, 1)
(20, 2)


## 9. Interview Signals
Questions FAANG asks:
- Why does `zip` stop at the shortest iterable?
- When is that behavior useful and when is it risky?
- How does `zip` compare to manual index-based iteration?
- How would you guard against silent truncation?

## 10. Exercise (Non-trivial)
Design a CSV row-to-dictionary builder that pairs headers to values, validates equal lengths, reports truncation clearly, and decides whether missing or extra columns should be rejected, defaulted, or logged.

In [7]:
def build_record(headers, row):
    return dict(zip(headers, row))

# Task:
# 1. Detect unequal lengths.
# 2. Decide whether to fail fast or recover.
# 3. Explain why silent truncation is dangerous here.